In [ ]:
import os, json
# Base-vs-student tool-use accuracy for the SeqKD distillation recipe. Task success rate (ASR) is the
# headline metric; MMLU + non-thinking IFEval (via sagemaker_code/eval_lmeval.sh) guard against regression.

## 0.0 Update the system prompt for your task and tools

In [ ]:
system_prompt = (
    "You are a shopping assistant. Use the available tools to satisfy the request: "
    "search the catalog, inspect candidates, verify the constraints, then recommend."
)
# Held-out ShoppingBench eval problems (query + tools + gold product/criteria).
# eval_dataset = load_dataset(...)  # your held-out split

## 1.0 Tool-use ASR — Success Rate on the eval set

The ASR rollout + scoring wiring lives in **`sagemaker_code/shopping_asr_eval.sh`**: it serves the merged model with vLLM (one Qwen3.5 instance per GPU), runs the ShoppingBench ReAct rollout (`src/agent/run_rollout.py`) against the search server, then scores the trajectories with the ORM (`src/agent/run_evaluate.py`, success rate = rule≥1).

Prerequisites (this harness *is* ShoppingBench, not vendored here): the ShoppingBench repo at `$SB` (providing `src/agent/run_rollout.py` + `run_evaluate.py` + a `config/rollout/<TAG>_qwen35-4b.json`), the search server (`:5631`), a JDK at `$JAVA_HOME`, and the merged dir carrying `preprocessor_config.json` + `processor_config.json`.

In [ ]:
import subprocess, re, pathlib

def run_asr_eval(merged_dir, served_name, rollout_cfg, gpu=0, port=1078, iface="enp75s0"):
    """Serve + ReAct rollout + ORM scoring via sagemaker_code/shopping_asr_eval.sh; parse ASR from stdout.
    Requires the ShoppingBench harness + search server (see the markdown above)."""
    script = pathlib.Path("sagemaker_code/shopping_asr_eval.sh")
    out = subprocess.run(
        ["bash", str(script), merged_dir, served_name, rollout_cfg, str(gpu), str(port), iface],
        capture_output=True, text=True)
    print(out.stdout[-2000:])
    if out.returncode != 0:
        print("ASR eval failed:", out.stderr[-1000:]); return None
    m = re.search(r"success[^0-9]*([01]?\.\d+)", out.stdout, re.I)
    return {"asr": float(m.group(1)) if m else None}

# tuned = run_asr_eval("./merged", "shop-r64", "config/rollout/r64_qwen35-4b.json", gpu=0)
# base  = run_asr_eval("Qwen/Qwen3.5-4B", "base", "config/rollout/base_qwen35-4b.json", gpu=0)

#### Student (SeqKD LoRA r64) vs base

In [ ]:
# tuned_metrics = generate_model_metrics("./merged", eval_samples)   # from notebook 03
# base_metrics  = generate_model_metrics("Qwen/Qwen3.5-4B", eval_samples)
# Reference numbers from the recipe run (thinking-off, 250-problem eval):
tuned_metrics = {"asr": 0.624, "n": 250}
base_metrics  = {"asr": 0.432, "n": 250}
print(base_metrics, tuned_metrics)

## 2.0 Regression portfolio — MMLU + non-thinking IFEval

Run offline on a GPU box (the merged model must contain `preprocessor_config.json` + `processor_config.json`):
```bash
bash sagemaker_code/eval_lmeval.sh ./merged <NET_IFACE> 0 seqkd-r64
```
Prints MMLU acc + IFEval prompt/inst strict. Base gate: IFEval ~82.6 pp-strict (NOT ~28 — must be non-thinking), MMLU ~0.697.

In [ ]:
# Reference portfolio (base vs SeqKD r64 student):
portfolio = {
    "MMLU (acc)":            {"base": 0.697, "student": 0.697},
    "IFEval (pp-strict)":    {"base": 0.826, "student": 0.810},
    "Tool-use ASR":          {"base": 0.432, "student": 0.624},
}
import json; print(json.dumps(portfolio, indent=2))

## 3.0 Visualization

In [ ]:
import pandas as pd
df = pd.DataFrame(portfolio).T
df["delta"] = df["student"] - df["base"]
try:
    import matplotlib.pyplot as plt
    ax = df[['base', 'student']].plot(kind='bar', figsize=(8, 4))
    ax.set_title('SeqKD distillation: base vs student (Qwen3.5-4B)')
    ax.set_ylabel('score'); plt.xticks(rotation=15, ha='right'); plt.tight_layout(); plt.show()
except Exception as ex:
    print('(matplotlib unavailable)', ex)
df